<a href="https://colab.research.google.com/github/hbisgin/DeepLearning/blob/main/DL_15_EmbeddinOverview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Define vocabulary and mapping

In [1]:
import torch
vocab = ["cat", "dog", "bird", "fish"]
stoi = {w: i for i, w in enumerate(vocab)}

print("Vocabulary:", vocab)
print("stoi:", stoi)


Vocabulary: ['cat', 'dog', 'bird', 'fish']
stoi: {'cat': 0, 'dog': 1, 'bird': 2, 'fish': 3}


2. Define embedding matrix (V x D)

In [2]:
E = torch.tensor([
    [0.2, 0.1, 0.5],   # cat
    [0.0, 0.3, 0.7],   # dog
    [0.6, 0.4, 0.2],   # bird
    [0.9, 0.8, 0.1]    # fish
], dtype=torch.float)

print("\nEmbedding matrix E:\n", E)
print("Shape of E:", E.shape)  # (vocab_size, embedding_dim)


Embedding matrix E:
 tensor([[0.2000, 0.1000, 0.5000],
        [0.0000, 0.3000, 0.7000],
        [0.6000, 0.4000, 0.2000],
        [0.9000, 0.8000, 0.1000]])
Shape of E: torch.Size([4, 3])


3. One-hot encode a single word

In [3]:
word = "dog"
idx = stoi[word]

one_hot = torch.zeros(len(vocab))
one_hot[idx] = 1

print("\nWord:", word)
print("Index:", idx)
print("One-hot vector:", one_hot)


Word: dog
Index: 1
One-hot vector: tensor([0., 1., 0., 0.])


Verify: same as direct row lookup and Encode multiple words (sequence)

In [4]:

embedding = one_hot @ E

print("\nEmbedding of '{}':".format(word), embedding)



print("Direct lookup E[idx]:", E[idx])

############

sentence = ["cat", "fish", "dog"]
indices = [stoi[w] for w in sentence]

print("\nSentence:", sentence)
print("Indices:", indices)


Embedding of 'dog': tensor([0.0000, 0.3000, 0.7000])
Direct lookup E[idx]: tensor([0.0000, 0.3000, 0.7000])

Sentence: ['cat', 'fish', 'dog']
Indices: [0, 3, 1]


In [5]:
# Convert to one-hot matrix

one_hots = torch.eye(len(vocab))[indices]

print("\nOne-hot matrix:\n", one_hots)


# Compute embeddings for all words

embeddings = one_hots @ E

print("\nEmbeddings for sentence:\n", embeddings)


# Finally

print("\nKey idea:")
print("Embedding = one-hot × embedding matrix")
print("This is equivalent to selecting rows from E")


One-hot matrix:
 tensor([[1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 1., 0., 0.]])

Embeddings for sentence:
 tensor([[0.2000, 0.1000, 0.5000],
        [0.9000, 0.8000, 0.1000],
        [0.0000, 0.3000, 0.7000]])

Key idea:
Embedding = one-hot × embedding matrix
This is equivalent to selecting rows from E


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# toy sentence
text = "i like deep learning"
words = text.split()

# vocab
vocab = sorted(list(set(words)))
stoi = {w:i for i,w in enumerate(vocab)}
vocab_size = len(vocab)

# convert to indices
sequence = torch.tensor([stoi[w] for w in words])

# inputs and targets
X = sequence[:-1]   # [i, like, deep]
Y = sequence[1:]    # [like, deep, learning]

In [7]:
class NextWordModel(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.E = nn.Parameter(torch.randn(vocab_size, emb_dim))
        self.linear = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        emb = self.E[x]       # (T, D)
        out = self.linear(emb)
        return out

In [8]:
model = NextWordModel(vocab_size, emb_dim=5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    logits = model(X)
    loss = criterion(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(loss.item())

1.3198176622390747
0.28476110100746155
0.11638471484184265
0.06465845555067062
0.042828869074583054


In [9]:
embeddings = model.E.detach()

for i, word in enumerate(vocab):
    print(word, embeddings[i])

deep tensor([ 0.5950, -0.0879, -0.6928, -0.9513,  1.5384])
i tensor([-0.4935, -2.0512,  0.7900, -1.4863, -0.8742])
learning tensor([ 1.3280,  0.8909, -0.4843,  0.8323, -0.9647])
like tensor([-0.6869,  2.3025,  0.7735,  1.4949,  1.3954])


In [10]:
model.linear.weight

Parameter containing:
tensor([[-0.0694,  1.0456,  0.5450,  0.5586,  0.0412],
        [ 0.2149, -0.1283, -0.4153,  0.2927, -0.2137],
        [ 0.5297, -0.1579, -0.8366, -0.5669,  0.8567],
        [-0.3767, -0.8373,  0.5438, -0.4471, -0.8350]], requires_grad=True)

In [11]:
# toy corpus
text = "i like deep learning and i like pytorch"
words = text.split()

# build vocab
vocab = list(set(words))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}
vocab_size = len(vocab)


print("Vocabulary:", vocab)
print("stoi:", stoi)

Vocabulary: ['deep', 'learning', 'and', 'like', 'i', 'pytorch']
stoi: {'deep': 0, 'learning': 1, 'and': 2, 'like': 3, 'i': 4, 'pytorch': 5}


#Using window approach (skip-gram) to predict the previous and next word

In [12]:

window_size = 1
pairs = []
for i, word in enumerate(words):
    for j in range(-window_size, window_size + 1):
        if j == 0 or i + j < 0 or i + j >= len(words):
            continue
        center = stoi[word]
        context = stoi[words[i + j]]
        pairs.append((center, context))

print("\nTraining pairs (as indices):")
print(pairs)

X = torch.tensor([p[0] for p in pairs], dtype=torch.long)
Y = torch.tensor([p[1] for p in pairs], dtype=torch.long)

print("\nX:", X)
print("Y:", Y)
print("len(X):", len(X), "len(Y):", len(Y))


Training pairs (as indices):
[(4, 3), (3, 4), (3, 0), (0, 3), (0, 1), (1, 0), (1, 2), (2, 1), (2, 4), (4, 2), (4, 3), (3, 4), (3, 5), (5, 3)]

X: tensor([4, 3, 3, 0, 0, 1, 1, 2, 2, 4, 4, 3, 3, 5])
Y: tensor([3, 4, 0, 3, 1, 0, 2, 1, 4, 2, 3, 4, 5, 3])
len(X): 14 len(Y): 14


In [13]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.E = nn.Parameter(torch.randn(vocab_size, embedding_dim))
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        emb = self.E[x]          # embedding lookup
        out = self.linear(emb)   # predict context
        return out

In [14]:
import torch.optim as optim

model = Word2Vec(vocab_size, embedding_dim=5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    logits = model(X)
    loss = criterion(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.8443
Epoch 20, Loss: 1.3340
Epoch 40, Loss: 1.1761
Epoch 60, Loss: 1.0913
Epoch 80, Loss: 1.0332


In [15]:
embeddings = model.E.detach()

for i, word in enumerate(vocab):
    print(word, embeddings[i])

deep tensor([ 1.5665,  0.7521, -1.8096, -0.5212,  2.9553])
learning tensor([-0.8973, -0.1982, -2.4179,  2.4304, -0.7459])
and tensor([-0.4655, -1.6301,  0.1965,  1.2788, -0.7060])
like tensor([-0.2031, -0.8001,  0.2137, -0.2755, -0.8562])
i tensor([ 0.4905, -0.3349,  1.3835,  2.0812,  0.6714])
pytorch tensor([ 0.8651,  1.2681,  2.5603, -1.4855,  1.2608])


In [16]:
def similarity(w1, w2):
    v1 = embeddings[stoi[w1]]
    v2 = embeddings[stoi[w2]]
    return torch.cosine_similarity(v1, v2, dim=0)

print("like vs i:", similarity("like", "i"))

like vs i: tensor(-0.2080)
